In [1]:
import matplotlib.pyplot as plt
from collections import Counter
import pandas as pd

In [2]:
def get_number_pocket_pairs(df):
    return len(set(tuple(sorted([i, j])) for i, j in zip(df['Pocket1'], df['Pocket2'])))

def get_number_protein_pairs(df):
    return len(set(tuple(sorted([i, j])) for i, j in zip(df['Protein1'], df['Protein2'])))

In [3]:
#####################################################
#####################################################
###############------ PHASE 1 ------#################
#####################################################
#####################################################

In [4]:
###########  PAIRS  ###########

In [5]:
ALL_RESULTS_PAIRS = pd.read_csv("../processed/protein_prioritization/pairs.tsv", sep='\t', low_memory=False)
ALL_RESULTS_PAIRS["InterPro-Pocket1"] = ALL_RESULTS_PAIRS["InterPro-Pocket1"].fillna("").astype(str)
ALL_RESULTS_PAIRS["InterPro-Pocket2"] = ALL_RESULTS_PAIRS["InterPro-Pocket2"].fillna("").astype(str)

proteins = sorted(set(ALL_RESULTS_PAIRS['Protein1']).union(set(ALL_RESULTS_PAIRS['Protein2'])))

In [6]:
ALL_RESULTS_PAIRS

,Protein1,Pocket1,InterPro-Pocket1,Protein2,Pocket2,InterPro-Pocket2,PocketVec distance,Protein RMSD,Protein SEQ ID (CO),Protein SEQ ID (NW),Dist PDB 1,Dist PDB 2,Dist Alphafill 1,Dist AlphaFill 2,P2Rank score 1,P2Rank score 2
0,P9WFT5,alphafold3_P9WFT5_model_2_pocket_2,Anticodon Binding Domain;Catalytic Domain (ATP...,P9WFU3,alphafold2_P9WFU3_model_0_pocket_1,,0.114,4.39,35.3659,20.40,NaN,NaN,19.860,1.487,7.45,34.45
1,P9WFT1,alphafold2_P9WFT1_model_0_pocket_2,Catalytic Domain (ATP Binding Site),P9WFU5,alphafold3_P9WFU5_model_2_pocket_1,Catalytic Domain (ATP Binding Site);Other too ...,0.117,18.21,29.0055,12.29,NaN,NaN,10.547,0.762,5.96,17.36
2,P9WFS9,chai1_P9WFS9_model_4_pocket_5,Catalytic Domain (ATP Binding Site);Editing Do...,P9WFV3,alphafold3_P9WFV3_model_4_pocket_2,Catalytic Domain (ATP Binding Site);Editing Do...,0.118,2.60,30.8422,25.15,NaN,NaN,0.383,4.391,9.76,20.01
3,P9WFU5,alphafold3_P9WFU5_model_2_pocket_1,Catalytic Domain (ATP Binding Site);Other too ...,P9WFV1,alphafold2_P9WFV1_model_0_pocket_4,Catalytic Domain (ATP Binding Site),0.121,1.97,32.9167,26.00,NaN,NaN,0.762,15.104,17.36,5.56
4,P9WFV7,swissmodel_P9WFV7_model_1_pocket_2,Anticodon Binding Domain,P9WFW5,chai1_P9WFW5_model_4_pocket_2,Catalytic Domain (ATP Binding Site);Other too ...,0.121,17.16,25.0000,12.75,NaN,NaN,24.070,12.217,3.70,4.69
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,P9WFT5,chai1_P9WFT5_model_3_pocket_2,Anticodon Binding Domain;Catalytic Domain (ATP...,P9WFW1,swissmodel_P9WFW1_model_0_pocket_3,Catalytic Domain (ATP Binding Site),0.348,15.29,26.3736,11.64,NaN,NaN,20.842,4.442,6.47,5.31
32557,P9WFT3,alphafold3_P9WFT3_model_0_pocket_1,Catalytic Domain (ATP Binding Site),P9WFT5,alphafold3_P9WFT5_model_1_pocket_2,Anticodon Binding Domain;Catalytic Domain (ATP...,0.349,5.84,29.5732,8.57,NaN,NaN,0.414,20.690,10.19,5.66
32558,P9WFT5,alphafold3_P9WFT5_model_1_pocket_2,Anticodon Binding Domain;Catalytic Domain (ATP...,P9WFW1,swissmodel_P9WFW1_model_0_pocket_3,Catalytic Domain (ATP Binding Site),0.367,15.74,26.3736,11.64,NaN,NaN,20.690,4.442,5.66,5.31
32559,P9WFT5,chai1_P9WFT5_model_0_pocket_3,Anticodon Binding Domain;Catalytic Domain (ATP...,P9WFW1,swissmodel_P9WFW1_model_0_pocket_3,Catalytic Domain (ATP Binding Site),0.368,14.82,26.3736,11.64,NaN,NaN,22.371,4.442,8.72,5.31


In [7]:
RMSD_CUT_OFF = 10
SEQID_CUT_OFF = 30

In [8]:
# CATALYTIC COUNTS
CATALYTIC_COUNTS = []
for i,j in zip(ALL_RESULTS_PAIRS['InterPro-Pocket1'], ALL_RESULTS_PAIRS['InterPro-Pocket2']):
    counts = []
    for interpro in [i,j]:
        if "Catalytic Domain (ATP Binding Site)" in interpro:
            counts.append(1)
        else:
            counts.append(0)
    CATALYTIC_COUNTS.append(sum(counts))
ALL_RESULTS_PAIRS['catalytic_counts'] = CATALYTIC_COUNTS


# GLOBAL SIMILARITY
GLOBAL_SIMILARITY = []
for seqid, rmsd in zip(ALL_RESULTS_PAIRS['Protein SEQ ID (CO)'], ALL_RESULTS_PAIRS['Protein RMSD']):
    counts = []
    if seqid > SEQID_CUT_OFF or rmsd < RMSD_CUT_OFF:
        counts.append(1)
    else:
        counts.append(0)
    GLOBAL_SIMILARITY.append(sum(counts))
ALL_RESULTS_PAIRS['global_similarity'] = GLOBAL_SIMILARITY

In [9]:
TOTAL = []
PROTEIN_TO_OCCURRENCES_ALL = {i: {} for i in proteins}

GS, CAT = 0, 0
COND = (ALL_RESULTS_PAIRS['global_similarity'] == GS) & (ALL_RESULTS_PAIRS['catalytic_counts'] == CAT)
COND = ALL_RESULTS_PAIRS[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_pairs(COND):,}[{get_number_protein_pairs(COND)}]")
TOTAL.append(get_number_pocket_pairs(COND))
counter = Counter(COND['Protein1'].tolist() + COND['Protein2'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_ALL[protein][(GS, CAT)] = counter[protein]

GS, CAT = 0, 1
COND = (ALL_RESULTS_PAIRS['global_similarity'] == GS) & (ALL_RESULTS_PAIRS['catalytic_counts'] == CAT)
COND = ALL_RESULTS_PAIRS[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_pairs(COND):,}[{get_number_protein_pairs(COND)}]")
TOTAL.append(get_number_pocket_pairs(COND))
counter = Counter(COND['Protein1'].tolist() + COND['Protein2'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_ALL[protein][(GS, CAT)] = counter[protein]

GS, CAT = 0, 2
COND = (ALL_RESULTS_PAIRS['global_similarity'] == GS) & (ALL_RESULTS_PAIRS['catalytic_counts'] >= CAT)
COND = ALL_RESULTS_PAIRS[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_pairs(COND):,}[{get_number_protein_pairs(COND)}]")
TOTAL.append(get_number_pocket_pairs(COND))
counter = Counter(COND['Protein1'].tolist() + COND['Protein2'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_ALL[protein][(GS, CAT)] = counter[protein]

GS, CAT = 1, 0
COND = (ALL_RESULTS_PAIRS['global_similarity'] == GS) & (ALL_RESULTS_PAIRS['catalytic_counts'] == CAT)
COND = ALL_RESULTS_PAIRS[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_pairs(COND):,}[{get_number_protein_pairs(COND)}]")
TOTAL.append(get_number_pocket_pairs(COND))
counter = Counter(COND['Protein1'].tolist() + COND['Protein2'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_ALL[protein][(GS, CAT)] = counter[protein]

GS, CAT = 1, 1
COND = (ALL_RESULTS_PAIRS['global_similarity'] == GS) & (ALL_RESULTS_PAIRS['catalytic_counts'] == CAT)
COND = ALL_RESULTS_PAIRS[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_pairs(COND):,}[{get_number_protein_pairs(COND)}]")
TOTAL.append(get_number_pocket_pairs(COND))
counter = Counter(COND['Protein1'].tolist() + COND['Protein2'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_ALL[protein][(GS, CAT)] = counter[protein]

GS, CAT = 1, 2
COND = (ALL_RESULTS_PAIRS['global_similarity'] == GS) & (ALL_RESULTS_PAIRS['catalytic_counts'] >= CAT)
COND = ALL_RESULTS_PAIRS[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_pairs(COND):,}[{get_number_protein_pairs(COND)}]")
TOTAL.append(get_number_pocket_pairs(COND))
counter = Counter(COND['Protein1'].tolist() + COND['Protein2'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_ALL[protein][(GS, CAT)] = counter[protein]

GS 0 -- CAT 0 -- 1,069[25]
GS 0 -- CAT 1 -- 5,288[78]
GS 0 -- CAT 2 -- 4,162[37]
GS 1 -- CAT 0 -- 3,057[47]
GS 1 -- CAT 1 -- 10,778[96]
GS 1 -- CAT 2 -- 8,207[57]


In [10]:
sum(TOTAL)

32561

In [11]:
ALL_RESULTS_PAIRS_POCKETVEC_017 = ALL_RESULTS_PAIRS[(ALL_RESULTS_PAIRS['PocketVec distance'] < 0.17)].reset_index(drop=True)

In [12]:
len(ALL_RESULTS_PAIRS_POCKETVEC_017)

1481

In [13]:
TOTAL = []
PROTEIN_TO_OCCURRENCES_POCKETVEC_017 = {i: {} for i in proteins}

GS, CAT = 0, 0
COND = (ALL_RESULTS_PAIRS_POCKETVEC_017['global_similarity'] == GS) & (ALL_RESULTS_PAIRS_POCKETVEC_017['catalytic_counts'] == CAT)
COND = ALL_RESULTS_PAIRS_POCKETVEC_017[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_pairs(COND):,}[{get_number_protein_pairs(COND)}]")
TOTAL.append(get_number_pocket_pairs(COND))
counter = Counter(COND['Protein1'].tolist() + COND['Protein2'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_POCKETVEC_017[protein][(GS, CAT)] = counter[protein]

GS, CAT = 0, 1
COND = (ALL_RESULTS_PAIRS_POCKETVEC_017['global_similarity'] == GS) & (ALL_RESULTS_PAIRS_POCKETVEC_017['catalytic_counts'] == CAT)
COND = ALL_RESULTS_PAIRS_POCKETVEC_017[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_pairs(COND):,}[{get_number_protein_pairs(COND)}]")
TOTAL.append(get_number_pocket_pairs(COND))
counter = Counter(COND['Protein1'].tolist() + COND['Protein2'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_POCKETVEC_017[protein][(GS, CAT)] = counter[protein]

GS, CAT = 0, 2
COND = (ALL_RESULTS_PAIRS_POCKETVEC_017['global_similarity'] == GS) & (ALL_RESULTS_PAIRS_POCKETVEC_017['catalytic_counts'] >= CAT)
COND = ALL_RESULTS_PAIRS_POCKETVEC_017[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_pairs(COND):,}[{get_number_protein_pairs(COND)}]")
TOTAL.append(get_number_pocket_pairs(COND))
counter = Counter(COND['Protein1'].tolist() + COND['Protein2'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_POCKETVEC_017[protein][(GS, CAT)] = counter[protein]

GS, CAT = 1, 0
COND = (ALL_RESULTS_PAIRS_POCKETVEC_017['global_similarity'] == GS) & (ALL_RESULTS_PAIRS_POCKETVEC_017['catalytic_counts'] == CAT)
COND = ALL_RESULTS_PAIRS_POCKETVEC_017[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_pairs(COND):,}[{get_number_protein_pairs(COND)}]")
TOTAL.append(get_number_pocket_pairs(COND))
counter = Counter(COND['Protein1'].tolist() + COND['Protein2'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_POCKETVEC_017[protein][(GS, CAT)] = counter[protein]

GS, CAT = 1, 1
COND = (ALL_RESULTS_PAIRS_POCKETVEC_017['global_similarity'] == GS) & (ALL_RESULTS_PAIRS_POCKETVEC_017['catalytic_counts'] == CAT)
COND = ALL_RESULTS_PAIRS_POCKETVEC_017[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_pairs(COND):,}[{get_number_protein_pairs(COND)}]")
TOTAL.append(get_number_pocket_pairs(COND))
counter = Counter(COND['Protein1'].tolist() + COND['Protein2'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_POCKETVEC_017[protein][(GS, CAT)] = counter[protein]

GS, CAT = 1, 2
COND = (ALL_RESULTS_PAIRS_POCKETVEC_017['global_similarity'] == GS) & (ALL_RESULTS_PAIRS_POCKETVEC_017['catalytic_counts'] >= CAT)
COND = ALL_RESULTS_PAIRS_POCKETVEC_017[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_pairs(COND):,}[{get_number_protein_pairs(COND)}]")
TOTAL.append(get_number_pocket_pairs(COND))
counter = Counter(COND['Protein1'].tolist() + COND['Protein2'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_POCKETVEC_017[protein][(GS, CAT)] = counter[protein]

GS 0 -- CAT 0 -- 28[10]
GS 0 -- CAT 1 -- 194[48]
GS 0 -- CAT 2 -- 149[25]
GS 1 -- CAT 0 -- 69[27]
GS 1 -- CAT 1 -- 403[73]
GS 1 -- CAT 2 -- 638[42]


In [14]:
sum(TOTAL)

1481

In [15]:
ALL_RESULTS_PAIRS_POCKETVEC_014 = ALL_RESULTS_PAIRS[(ALL_RESULTS_PAIRS['PocketVec distance'] < 0.14)].reset_index(drop=True)

In [ ]:
len(ALL_RESULTS_PAIRS_POCKETVEC_014)

In [ ]:
TOTAL = []
PROTEIN_TO_OCCURRENCES_POCKETVEC_017_MORE = {i: {} for i in proteins}

GS, CAT = 0, 0
COND = (ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['global_similarity'] == GS) & (ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['catalytic_counts'] == CAT)
COND = ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_triplets(COND):,}[{get_number_protein_triplets(COND)}]")
TOTAL.append(get_number_pocket_triplets(COND))
counter = Counter(COND['protein1'].tolist() + COND['protein2'].tolist() + COND['protein3'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_POCKETVEC_017_MORE[protein][(GS, CAT)] = counter[protein]

GS, CAT = 0, 1
COND = (ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['global_similarity'] == GS) & (ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['catalytic_counts'] == CAT)
COND = ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_triplets(COND):,}[{get_number_protein_triplets(COND)}]")
TOTAL.append(get_number_pocket_triplets(COND))
counter = Counter(COND['protein1'].tolist() + COND['protein2'].tolist() + COND['protein3'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_POCKETVEC_017_MORE[protein][(GS, CAT)] = counter[protein]

GS, CAT = 0, 2
COND = (ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['global_similarity'] == GS) & (ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['catalytic_counts'] >= CAT)
COND = ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_triplets(COND):,}[{get_number_protein_triplets(COND)}]")
TOTAL.append(get_number_pocket_triplets(COND))
counter = Counter(COND['protein1'].tolist() + COND['protein2'].tolist() + COND['protein3'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_POCKETVEC_017_MORE[protein][(GS, CAT)] = counter[protein]

GS, CAT = 1, 0
COND = (ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['global_similarity'] == GS) & (ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['catalytic_counts'] == CAT)
COND = ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_triplets(COND):,}[{get_number_protein_triplets(COND)}]")
TOTAL.append(get_number_pocket_triplets(COND))
counter = Counter(COND['protein1'].tolist() + COND['protein2'].tolist() + COND['protein3'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_POCKETVEC_017_MORE[protein][(GS, CAT)] = counter[protein]

GS, CAT = 1, 1
COND = (ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['global_similarity'] == GS) & (ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['catalytic_counts'] == CAT)
COND = ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_triplets(COND):,}[{get_number_protein_triplets(COND)}]")
TOTAL.append(get_number_pocket_triplets(COND))
counter = Counter(COND['protein1'].tolist() + COND['protein2'].tolist() + COND['protein3'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_POCKETVEC_017_MORE[protein][(GS, CAT)] = counter[protein]

GS, CAT = 1, 2
COND = (ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['global_similarity'] == GS) & (ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['catalytic_counts'] >= CAT)
COND = ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_triplets(COND):,}[{get_number_protein_triplets(COND)}]")
TOTAL.append(get_number_pocket_triplets(COND))
counter = Counter(COND['protein1'].tolist() + COND['protein2'].tolist() + COND['protein3'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_POCKETVEC_017_MORE[protein][(GS, CAT)] = counter[protein]

GS, CAT = 2, 0
COND = (ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['global_similarity'] >= GS) & (ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['catalytic_counts'] == CAT)
COND = ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_triplets(COND):,}[{get_number_protein_triplets(COND)}]")
TOTAL.append(get_number_pocket_triplets(COND))
counter = Counter(COND['protein1'].tolist() + COND['protein2'].tolist() + COND['protein3'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_POCKETVEC_017_MORE[protein][(GS, CAT)] = counter[protein]

GS, CAT = 2, 1
COND = (ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['global_similarity'] >= GS) & (ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['catalytic_counts'] == CAT)
COND = ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_triplets(COND):,}[{get_number_protein_triplets(COND)}]")
TOTAL.append(get_number_pocket_triplets(COND))
counter = Counter(COND['protein1'].tolist() + COND['protein2'].tolist() + COND['protein3'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_POCKETVEC_017_MORE[protein][(GS, CAT)] = counter[protein]

GS, CAT = 2, 2
COND = (ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['global_similarity'] >= GS) & (ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['catalytic_counts'] >= CAT)
COND = ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE[COND].reset_index(drop=True)
print(f"GS {GS} -- CAT {CAT} -- {get_number_pocket_triplets(COND):,}[{get_number_protein_triplets(COND)}]")
TOTAL.append(get_number_pocket_triplets(COND))
counter = Counter(COND['protein1'].tolist() + COND['protein2'].tolist() + COND['protein3'].tolist())
for protein in proteins:
    PROTEIN_TO_OCCURRENCES_POCKETVEC_017_MORE[protein][(GS, CAT)] = counter[protein]

In [ ]:
sum(TOTAL)

In [ ]:
#####################################################
#####################################################
###############------ PHASE 2 ------#################
#####################################################
#####################################################

In [ ]:
#####################################################
#####################################################
###############------ PHASE 3 ------#################
#####################################################
#####################################################

In [ ]:
###########  TRIPLETS  ###########

In [ ]:
ERSILIA_COLORS = {(0,0): '#50285A', 
                  (0,1): '#FAD782', 
                  (0,2): '#FAA08B', 
                  (1,0): '#DC9FDC', 
                  (1,1): '#AA96FA', 
                  (1,2): '#8DC7FA', 
                  (2,0): '#BEE6B4', 
                  (2,1): '#D2D2D2', 
                  (2,2): 'w'}

inds = sorted(ERSILIA_COLORS)

In [ ]:
### ALL RESULTS ###

In [ ]:
proteins = sorted(PROTEIN_TO_OCCURRENCES_ALL, key=lambda x: sum(PROTEIN_TO_OCCURRENCES_ALL[x].values()), reverse=True)

for c1, protein in enumerate(proteins):
    for c2, ind in enumerate(inds):
        bottom = sum([PROTEIN_TO_OCCURRENCES_ALL[protein][i] for i in inds if i < ind])
        if c1 == 0:
            plt.bar(c1, PROTEIN_TO_OCCURRENCES_ALL[protein][ind], bottom=bottom, color=ERSILIA_COLORS[ind], 
                    edgecolor='black', linewidth=1, zorder=2, label=f"GS {ind[0] if ind[0] != 2 else '>1'}, CAT {ind[1] if ind[1] != 2 else '>1'}")
        else:
            plt.bar(c1, PROTEIN_TO_OCCURRENCES_ALL[protein][ind], bottom=bottom, color=ERSILIA_COLORS[ind], edgecolor='black', linewidth=1, zorder=2) 

plt.title("Pocket Triplets - All Results", pad=12)
plt.ylabel('Number of protein occurrences', labelpad=12)
plt.xticks([i for i in range(len(proteins))], proteins, rotation=90)
plt.grid(linestyle='--')
plt.legend(loc='upper right', framealpha=1, edgecolor='k', ncols=3, prop={'size': 9})
plt.show()

In [ ]:
### POCKETVEC < 0.17 ###

In [ ]:
proteins = sorted(PROTEIN_TO_OCCURRENCES_POCKETVEC_017, key=lambda x: sum(PROTEIN_TO_OCCURRENCES_POCKETVEC_017[x].values()), reverse=True)

for c1, protein in enumerate(proteins):
    for c2, ind in enumerate(inds):
        bottom = sum([PROTEIN_TO_OCCURRENCES_POCKETVEC_017[protein][i] for i in inds if i < ind])
        if c1 == 0:
            plt.bar(c1, PROTEIN_TO_OCCURRENCES_POCKETVEC_017[protein][ind], bottom=bottom, color=ERSILIA_COLORS[ind], 
                    edgecolor='black', linewidth=1, zorder=2, label=f"GS {ind[0] if ind[0] != 2 else '>1'}, CAT {ind[1] if ind[1] != 2 else '>1'}")
        else:
            plt.bar(c1, PROTEIN_TO_OCCURRENCES_POCKETVEC_017[protein][ind], bottom=bottom, color=ERSILIA_COLORS[ind], edgecolor='black', linewidth=1, zorder=2) 

plt.title("Pocket Triplets - All PocketVec distances < 0.17", pad=12)
plt.ylabel('Number of protein occurrences', labelpad=12)
plt.xticks([i for i in range(len(proteins))], proteins, rotation=90)
plt.grid(linestyle='--')
plt.legend(loc='upper right', framealpha=1, edgecolor='k', ncols=3, prop={'size': 9})
plt.show()

In [ ]:
### POCKETVEC < 0.17 & MORE ###

In [ ]:
proteins = sorted(PROTEIN_TO_OCCURRENCES_POCKETVEC_017_MORE, key=lambda x: sum(PROTEIN_TO_OCCURRENCES_POCKETVEC_017_MORE[x].values()), reverse=True)

for c1, protein in enumerate(proteins):
    for c2, ind in enumerate(inds):
        bottom = sum([PROTEIN_TO_OCCURRENCES_POCKETVEC_017_MORE[protein][i] for i in inds if i < ind])
        if c1 == 0:
            plt.bar(c1, PROTEIN_TO_OCCURRENCES_POCKETVEC_017_MORE[protein][ind], bottom=bottom, color=ERSILIA_COLORS[ind], 
                    edgecolor='black', linewidth=1, zorder=2, label=f"GS {ind[0] if ind[0] != 2 else '>1'}, CAT {ind[1] if ind[1] != 2 else '>1'}")
        else:
            plt.bar(c1, PROTEIN_TO_OCCURRENCES_POCKETVEC_017_MORE[protein][ind], bottom=bottom, color=ERSILIA_COLORS[ind], edgecolor='black', linewidth=1, zorder=2) 

plt.title("Pocket Triplets - All PocketVec distances < 0.17 & more", pad=12)
plt.ylabel('Number of protein occurrences', labelpad=12)
plt.xticks([i for i in range(len(proteins))], proteins, rotation=90)
plt.grid(linestyle='--')
plt.legend(loc='upper right', framealpha=1, edgecolor='k', ncols=3, prop={'size': 9})
plt.show()

In [ ]:
### NOW FROM THE DOMAIN PERSPECTIVE ... ###

In [ ]:
ERSILIA_COLORS = ['#50285A', '#FAD782', '#FAA08B', '#DC9FDC', '#AA96FA', '#8DC7FA', '#BEE6B4', '#D2D2D2'][1:]

In [ ]:
### ALL RESULTS ###

In [ ]:
interpro_domains = ALL_RESULTS_TRIPLETS['interpro_pocket1'].tolist() + ALL_RESULTS_TRIPLETS['interpro_pocket2'].tolist() + ALL_RESULTS_TRIPLETS['interpro_pocket3'].tolist()
interpro_domains = [j for i in interpro_domains for j in i.split(";") if j != ""]
interpro_domains = Counter(interpro_domains)

domains = sorted(interpro_domains, key = lambda x: interpro_domains[x], reverse=True)
for c1, domain in enumerate(domains):
    plt.bar(c1, interpro_domains[domain], color=ERSILIA_COLORS[c1], edgecolor='black', linewidth=1, zorder=2, label=domain)

plt.title("Pocket Triplets - All Results", pad=12)
plt.ylabel('Number of domain occurrences', labelpad=12)
plt.xticks([])
plt.grid(linestyle='--')
plt.legend(loc='upper right', framealpha=1, edgecolor='k', prop={'size': 8})
plt.show()

In [ ]:
### POCKETVEC < 0.17 ###

In [ ]:
interpro_domains = ALL_RESULTS_TRIPLETS_POCKETVEC_017['interpro_pocket1'].tolist() + ALL_RESULTS_TRIPLETS_POCKETVEC_017['interpro_pocket2'].tolist() + ALL_RESULTS_TRIPLETS_POCKETVEC_017['interpro_pocket3'].tolist()
interpro_domains = [j for i in interpro_domains for j in i.split(";") if j != ""]
interpro_domains = Counter(interpro_domains)

domains = sorted(interpro_domains, key = lambda x: interpro_domains[x], reverse=True)
for c1, domain in enumerate(domains):
    plt.bar(c1, interpro_domains[domain], color=ERSILIA_COLORS[c1], edgecolor='black', linewidth=1, zorder=2, label=domain)

plt.title("Pocket Triplets - All PocketVec distances < 0.17", pad=12)
plt.ylabel('Number of domain occurrences', labelpad=12)
plt.xticks([])
plt.grid(linestyle='--')
plt.legend(loc='upper right', framealpha=1, edgecolor='k', prop={'size': 8})
plt.show()

In [ ]:
### POCKETVEC < 0.17 & MORE ###

In [ ]:
interpro_domains = ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['interpro_pocket1'].tolist() + ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['interpro_pocket2'].tolist() + ALL_RESULTS_TRIPLETS_POCKETVEC_017_MORE['interpro_pocket3'].tolist()
interpro_domains = [j for i in interpro_domains for j in i.split(";") if j != ""]
interpro_domains = Counter(interpro_domains)

domains = sorted(interpro_domains, key = lambda x: interpro_domains[x], reverse=True)
for c1, domain in enumerate(domains):
    plt.bar(c1, interpro_domains[domain], color=ERSILIA_COLORS[c1], edgecolor='black', linewidth=1, zorder=2, label=domain)

plt.title("Pocket Triplets - All PocketVec distances < 0.17 & more", pad=12)
plt.ylabel('Number of domain occurrences', labelpad=12)
plt.xticks([])
plt.grid(linestyle='--')
plt.legend(loc='upper right', framealpha=1, edgecolor='k', prop={'size': 8})
plt.show()